# Create meeting minutes from an Audio file in Gradio

In [ ]:
# --- Install deps ---
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
# --- Imports ---

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
import torch
import threading
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, TextIteratorStreamer
from transformers import pipeline
import gradio as gr

In [ ]:
# --- Constants ---

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
# --- Authentication ---
# assuming you've already added HF_TOKEN in Google Colab's Secret

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Using Open Source for Transcription - Hugging Face Pipelines

In [ ]:
# --- Transcription Pipeline ---

transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device="cuda",
    torch_dtype=torch.float16,
    chunk_length_s=30,  # Handles long recordings/meetings
    stride_length_s=4,  # Overlap between chunks to avoid cut-off words
)


def transcribe_audio(audio_path: str):
    if not audio_path:
        return ""
    try:
        result = transcriber(audio_path)
        # Some pipeline versions return a dict, some a list of dicts
        if isinstance(result, dict):
            return result.get("text", "")
        elif isinstance(result, list) and len(result) > 0:
            return result[0].get("text", "")
        return str(result)
    except Exception as e:
        return f"Transcription error: {str(e)}"

In [ ]:
# --- Model Loading Logic ---
def load_model_and_tokenizer(model_id: str):
    """Loads tokenizer and causal LM directly to CUDA with 4-bit NF4 quantization."""
    # 1. Configure 4-bit quantization
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    # 2. Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 3. Load model directly onto CUDA with quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=quant_config, device_map="auto"
    )

    return tokenizer, model

In [ ]:
# --- Streaming Inference Logic ---
def stream_response(
    tokenizer,
    model,
    messages: list[dict],
    max_new_tokens: int = 1024,
    temperature: float = 0.7,
):
    """Generates a real-time text stream from the model using conversation history.

    Args:
        tokenizer: Pre-loaded Hugging Face tokenizer.
        model: Pre-loaded (quantized) Hugging Face model.
        messages: Conversation history, e.g.:
                  [{"role": "system", "content": "..."}, {"role": "user",
                  "content": "..."}]
        max_new_tokens: Maximum tokens to generate (default: 1024).
        temperature: Sampling temperature (default: 0.7).

    Yields:
        str: Incremental text chunks as they are generated.
    """
    # 1. Format messages using the model's native chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    # 2. Set up the streamer
    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, decode_kwargs={"skip_special_tokens": True}
    )

    # 3. Configure generation parameters
    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True if temperature > 0 else False,
    )

    # 4. Run model.generate in a background thread to prevent blocking
    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 5. Yield decoded chunks as they arrive from the thread
    for chunk in streamer:
        yield chunk

In [ ]:
# --- Model Initialization ---
tokenizer, model = load_model_and_tokenizer("Qwen/Qwen2.5-3B-Instruct")

In [ ]:
# --- Summarization Function ---
def summarize_to_chatbot(transcript_text: str, history: list):
    """Generates a meeting summary and action items from transcription text and updates chat history."""
    if not transcript_text.strip():
        return history

    user_prompt = (
        f"Transcript:\n{transcript_text}\n\n"
        "Please provide a concise summary and explicit action items with owners."
    )

    # Append user turn and blank assistant container
    history.append({"role": "user", "content": user_prompt})
    history.append({"role": "assistant", "content": ""})

    # Prepare system prompt + entire conversation history
    system_msg = {
        "role": "system",
        "content": (
            "You are an executive assistant. Summarize meetings and extract "
            "action items with their explicit owners (- [Owner]: Task)."
        )
    }
    messages = [system_msg] + history[:-1]

    # Stream the assistant response
    accumulated = ""
    for token_chunk in stream_response(tokenizer, model, messages):
        accumulated += token_chunk
        history[-1]["content"] = accumulated
        yield history


In [ ]:
def stream_response(tokenizer, model, messages: list[dict]):
    # 1. Clean message history
    clean_messages = []
    for m in messages:
        if isinstance(m, dict) and "role" in m and "content" in m:
            content = str(m["content"]) if m["content"] is not None else ""
            clean_messages.append({"role": m["role"], "content": content})

    # 2. Tokenize inputs
    inputs = tokenizer.apply_chat_template(
        clean_messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    # 3. Configure streamer to explicitly skip special tokens
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,          # Skips special tokens like <|im_end|>
        decode_kwargs={"skip_special_tokens": True}
    )

    # 4. Gather EOS token IDs so generation stops on <|im_end|>
    eos_ids = [tokenizer.eos_token_id]
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id and im_end_id != tokenizer.unk_token_id:
        eos_ids.append(im_end_id)

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True,
        eos_token_id=eos_ids,              # Halts right at the boundary
    )

    # 5. Generate in thread
    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 6. Stream and strip any residual tags
    for chunk in streamer:
        cleaned_chunk = chunk.replace("<|im_end|>", "").replace("<|endoftext|>", "")
        if cleaned_chunk:
            yield cleaned_chunk

In [ ]:
def handle_follow_up(user_message: str, history: list):
    if not user_message or not user_message.strip():
        yield "", history
        return

    # Append user question and blank assistant entry
    history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": ""}
    ]

    # First yield immediately clears the text box and shows the user prompt
    yield "", history

    # Build system prompt + all messages except the blank assistant one
    system_msg = {
        "role": "system",
        "content": "You are a helpful assistant answering questions about the meeting transcript."
    }
    messages = [system_msg] + history[:-1]

    # Stream into the blank assistant slot
    accumulated = ""
    for chunk in stream_response(tokenizer, model, messages):
        accumulated += chunk
        history[-1]["content"] = accumulated
        yield "", history

In [ ]:

# --- Gradio UI Layout and Logic ---
with gr.Blocks() as demo:
    gr.Markdown("### Meeting Assistant: Audio Upload & Follow-up Chat")

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath")
            transcribe_btn = gr.Button("Process Meeting", variant="primary")
            transcript_box = gr.Textbox(label="Raw Transcript", lines=8)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=450)
            msg_input = gr.Textbox(placeholder="Ask follow-up questions about this meeting...", label="Follow-up Chat")
            clear_btn = gr.Button("Clear Chat")

    # Chain 1: Audio -> Transcript -> Chatbot Summary
    transcribe_btn.click(
        fn=transcribe_audio,
        inputs=[audio_input],
        outputs=[transcript_box]
    ).then(
        fn=summarize_to_chatbot,
        inputs=[transcript_box, chatbot],
        outputs=[chatbot]
    )

    # Chain 2: Follow-up questions in the Chatbot (REPLACED EVENT)
    msg_input.submit(
        fn=add_user_message,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot],
        queue=False
    ).then(
        fn=stream_follow_up,
        inputs=[chatbot],
        outputs=[chatbot]
    )

    clear_btn.click(lambda: [], None, chatbot, queue=False)
    
gr.close_all()
demo.launch(share=True)